# 09 · A full Transformer block (PyTorch)

Now assemble everything from 04–08 into one real, runnable block using PyTorch.

**One block = pre-norm multi-head attention (with residual) + pre-norm feed-forward (with
residual).** Stack `N` of these, add embeddings in front and an output head behind, and you
have a GPT.
```
x ─► LayerNorm ─► Multi-head Attn ─►(+x)─► LayerNorm ─► FFN ─►(+)─► out
```

In [1]:
import torch, torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                  nn.Linear(d_ff, d_model))

    def forward(self, x, attn_mask=None):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=attn_mask)   # self-attention: query=key=value=h
        x = x + a                                        # residual 1
        x = x + self.ff(self.ln2(x))                     # residual 2
        return x

torch.manual_seed(0)
d_model, n_heads, seq_len = 32, 4, 6
block = TransformerBlock(d_model, n_heads, d_ff=4*d_model)

x = torch.randn(1, seq_len, d_model)                                  # (batch, seq, d_model)
causal = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()  # True above diag = blocked
y = block(x, attn_mask=causal)
print("in", tuple(x.shape), "-> out", tuple(y.shape), "(identical -> stackable)")
assert y.shape == x.shape
print("parameters in one block:", sum(p.numel() for p in block.parameters()))

in (1, 6, 32) -> out (1, 6, 32) (identical -> stackable)
parameters in one block: 12704


In [2]:
# Stack N blocks: output of one feeds the next. This is literally a mini GPT body.
import torch.nn as nn
N = 4
gpt_body = nn.ModuleList([TransformerBlock(d_model, n_heads, 4*d_model) for _ in range(N)])
h = x
for blk in gpt_body:
    h = blk(h, attn_mask=causal)
print(f"after {N} stacked blocks:", tuple(h.shape))
assert h.shape == x.shape

after 4 stacked blocks: (1, 6, 32)


**Takeaway.** A block is just attention + FFN wrapped in norms and residuals. Stacking is
trivial because input and output shapes match. Next: **10 · training objective** — how these
weights actually get learned.